# Wholesale Operations Snapshot

## Business question

Which products and periods concentrate revenue, and how much do cancellations affect the business?

## Data source

UCI Online Retail dataset: real transactions from a UK-based online retailer between 2010 and 2011.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
data_path = Path("data/raw/Online Retail.xlsx")

data_path.exists()

True

In [3]:
df = pd.read_excel(data_path, nrows=10_000)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    10000 non-null  object        
 1   StockCode    10000 non-null  object        
 2   Description  9958 non-null   str           
 3   Quantity     10000 non-null  int64         
 4   InvoiceDate  10000 non-null  datetime64[us]
 5   UnitPrice    10000 non-null  float64       
 6   CustomerID   7709 non-null   float64       
 7   Country      10000 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 625.1+ KB


In [5]:
missing_values = df.isna().sum()
missing_percentage = (df.isna().mean() * 100).round(2)

pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage,
})

,missing_count,missing_percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,42,0.42
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,2291,22.91
Country,0,0.00


In [6]:
quality_checks = pd.Series({
    "rows": len(df),
    "negative_quantity_rows": (df["Quantity"] < 0).sum(),
    "zero_quantity_rows": (df["Quantity"] == 0).sum(),
    "negative_price_rows": (df["UnitPrice"] < 0).sum(),
    "zero_price_rows": (df["UnitPrice"] == 0).sum(),
    "cancelled_invoice_rows": df["InvoiceNo"].astype(str).str.startswith("C").sum(),
})

quality_checks

rows                      10000
negative_quantity_rows      130
zero_quantity_rows            0
negative_price_rows           0
zero_price_rows              46
cancelled_invoice_rows      100
dtype: int64

In [7]:
df["Quantity"] < 0

0       False
1       False
2       False
3       False
4       False
        ...  
9995    False
9996    False
9997    False
9998    False
9999    False
Name: Quantity, Length: 10000, dtype: bool

In [8]:
total_filas = len(df)
total_filas

10000

In [9]:
negative_quantity = df["Quantity"] < 0

df[negative_quantity].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [10]:
negative_quantity = df["Quantity"] < 0

In [11]:
negative_quantity.sum()

np.int64(130)

In [12]:
df[negative_quantity]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
8685,C537143,20685,DOORMAT RED RETROSPOT,-1,2010-12-05 12:58:00,7.95,12748.0,United Kingdom
8952,C537157,35953,FOLKART STAR CHRISTMAS DECORATIONS,-24,2010-12-05 13:09:00,1.25,15880.0,United Kingdom
9038,C537164,D,Discount,-1,2010-12-05 13:21:00,29.29,14527.0,United Kingdom
9493,C537203,37449,CERAMIC CAKE STAND + HANGING CAKES,-1,2010-12-05 14:44:00,9.95,14606.0,United Kingdom


In [13]:
negative_quantity_rows = df[negative_quantity]

In [14]:
negative_quantity_rows[["InvoiceNo", "Quantity"]].head(10)

,InvoiceNo,Quantity
141,C536379,-1
154,C536383,-1
235,C536391,-12
236,C536391,-24
237,C536391,-24
238,C536391,-24
239,C536391,-12
240,C536391,-12
241,C536391,-24
939,C536506,-6


In [15]:
cancelled_invoice = df["InvoiceNo"].astype(str).str.startswith("C")

In [16]:
cancelled_invoice.sum()

np.int64(100)

In [17]:
pd.crosstab(negative_quantity, cancelled_invoice)

InvoiceNo,False,True
Quantity,,
False,9870,0
True,30,100


In [18]:
negative_without_cancellation = negative_quantity & ~cancelled_invoice

In [19]:
negative_without_cancellation.sum()

np.int64(30)

In [20]:
exception_rows = df[negative_without_cancellation]

exception_rows[[
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "CustomerID",
]].head(30)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
2406,536589,21777,NaN,-10,0.0,NaN
4347,536764,84952C,NaN,-38,0.0,NaN
7188,536996,22712,NaN,-20,0.0,NaN
7189,536997,22028,NaN,-20,0.0,NaN
7190,536998,85067,NaN,-6,0.0,NaN
7192,537000,21414,NaN,-22,0.0,NaN
7193,537001,21653,NaN,-6,0.0,NaN
7195,537003,85126,NaN,-2,0.0,NaN
7196,537004,21814,NaN,-30,0.0,NaN
7197,537005,21692,NaN,-70,0.0,NaN


In [21]:
exception_rows[["InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice"]].sort_values("InvoiceDate")

,InvoiceNo,InvoiceDate,Quantity,UnitPrice
2406,536589,2010-12-01 16:50:00,-10,0.0
4347,536764,2010-12-02 14:42:00,-38,0.0
7188,536996,2010-12-03 15:30:00,-20,0.0
7189,536997,2010-12-03 15:30:00,-20,0.0
7190,536998,2010-12-03 15:30:00,-6,0.0
7192,537000,2010-12-03 15:32:00,-22,0.0
7193,537001,2010-12-03 15:33:00,-6,0.0
7195,537003,2010-12-03 15:33:00,-2,0.0
7196,537004,2010-12-03 15:34:00,-30,0.0
7197,537005,2010-12-03 15:35:00,-70,0.0


In [22]:
exception_rows["InvoiceDate"].dt.date.value_counts().sort_index()

InvoiceDate
2010-12-01     1
2010-12-02     1
2010-12-03    28
Name: count, dtype: int64